# Домашнее задание. Классификация изображений

Сегодня вам предстоить помочь телекомпании FOX в обработке их контента. Как вы знаете, сериал "Симпсоны" идет на телеэкранах более 25 лет, и за это время скопилось очень много видеоматериала. Персоонажи менялись вместе с изменяющимися графическими технологиями, и Гомер Симпсон-2018 не очень похож на Гомера Симпсона-1989. В этом задании вам необходимо классифицировать персонажей, проживающих в Спрингфилде. Думаю, нет смысла представлять каждого из них в отдельности.



В нашем тесте будет 991 картинка, для которых вам будет необходимо предсказать класс.


## Шаг 1. Установка зависимостей


####Установим необходимые библиотеки и проверим доступность CUDA


In [ ]:
# we will verify that GPU is enabled for this notebook
# following should print: CUDA is available!  Training on GPU ...
#
# if it prints otherwise, then you need to enable GPU:
# from Menu > Runtime > Change Runtime Type > Hardware Accelerator > GPU

import torch
import numpy as np
from torch.utils.tensorboard import SummaryWriter

train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')

writer = SummaryWriter("runs/exp1")

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi


In [ ]:
import pickle
import numpy as np
from skimage import io

from tqdm import tqdm, tqdm_notebook
from PIL import Image
from pathlib import Path

from torchvision import transforms, models
from torchvision.transforms import v2
from torchvision.models import ResNet18_Weights

import torchsummary

from multiprocessing.pool import ThreadPool
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn as nn
import torch.nn.functional as F

from matplotlib import colors, pyplot as plt
%matplotlib inline

import json
import pandas as pd

import os

import warnings
warnings.filterwarnings(action='ignore', category=DeprecationWarning)


 #### Определим константы, которые будем использовать в по ходу ноутбука


In [ ]:
# Режимы датасета
DATA_MODES = ["train", "val", "test"]

# Устройство для обучения
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Данные после распаковки архива Kaggle в Colab
TRAIN_DIR = Path("/content/train")
TEST_DIR = Path("/content/testset")

# Единый корень проекта на Google Drive. Все артефакты пишем только сюда.
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/simpsons")
CHECKPOINT_DIR = DRIVE_PROJECT_DIR / "checkpoints"
LABEL_AUDIT_DIR = DRIVE_PROJECT_DIR / "label_audit"

EFFICIENTNET_STAGE1_DIR = CHECKPOINT_DIR / "efficientnet_v2_s_stage1"
EFFICIENTNET_STAGE2_DIR = CHECKPOINT_DIR / "efficientnet_v2_s_stage2"
BEST_MODEL_PATH = EFFICIENTNET_STAGE2_DIR / "best_model.pth"

for directory in [DRIVE_PROJECT_DIR, CHECKPOINT_DIR, LABEL_AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

import sys
if str(DRIVE_PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_PROJECT_DIR))

# CSV с ручной проверкой подозрительных меток. Создается ниже в блоке аудита.
REVIEW_PATH = LABEL_AUDIT_DIR / "suspicious_manual.csv"

# Параметры нормировки ImageNet-весов
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

# Размер входа модели
RESCALE_SIZE = [224, 224]


## Шаг 2. Загрузка и обработка данных


#### Скачаем изображения по ссылке


In [ ]:
!gdown 1RxBQiZgRAfio2tWhEE7lzZ6IaJzLheH1


In [ ]:
!unzip -q /content/journey-springfield.zip


In [ ]:
# Опциональный блок исправления меток по CSV после ручной проверки.
# По умолчанию ничего не переносим: сначала нужно заполнить REVIEW_PATH и включить APPLY_LABEL_FIXES.
import shutil

APPLY_LABEL_FIXES = False


def move_reviewed_files_to_class(review_df, dry_run=True):
    """
    Переносит только строки с action == "MOVE".
    Целевой класс берется из correct_label, если он заполнен, иначе из predicted_label.
    """
    required_columns = ["path", "predicted_label", "action"]
    missing_columns = [col for col in required_columns if col not in review_df.columns]
    if missing_columns:
        raise ValueError(f"В review_df не хватает колонок: {missing_columns}")

    actions = review_df["action"].fillna("").astype(str).str.upper()
    rows_to_move = review_df[actions == "MOVE"].copy()
    print("Файлов к переносу:", len(rows_to_move))

    moved_paths = {}

    for _, row in rows_to_move.iterrows():
        src_path = Path(row["path"])
        correct_label = row.get("correct_label", "")
        target_class = "" if pd.isna(correct_label) else str(correct_label).strip()
        if not target_class:
            target_class = str(row["predicted_label"]).strip()

        if not src_path.exists():
            print("Файл не найден, пропускаю:", src_path)
            continue

        current_class_dir = src_path.parent
        split_root_dir = current_class_dir.parent
        dst_dir = split_root_dir / target_class
        dst_path = dst_dir / src_path.name

        if not dst_dir.exists():
            print("Папка целевого класса не найдена, пропускаю:", dst_dir)
            continue

        if current_class_dir.name == target_class:
            print("Файл уже находится в нужной папке, пропускаю:", src_path)
            continue

        if dst_path.exists():
            stem = src_path.stem
            suffix = src_path.suffix
            counter = 1
            while dst_path.exists():
                dst_path = dst_dir / f"{stem}_moved_{counter}{suffix}"
                counter += 1

        print("FROM:", src_path)
        print("TO:  ", dst_path)

        if not dry_run:
            shutil.move(str(src_path), str(dst_path))
            moved_paths[src_path] = dst_path

    if dry_run:
        print("Это был dry_run=True. Файлы НЕ были перенесены.")
    else:
        print("Готово. Файлы перенесены.")

    return moved_paths


In [ ]:
if APPLY_LABEL_FIXES:
    if not REVIEW_PATH.exists():
        raise FileNotFoundError(f"Файл ручной проверки не найден: {REVIEW_PATH}")

    review_df = pd.read_csv(REVIEW_PATH)
    moved_paths = move_reviewed_files_to_class(review_df=review_df, dry_run=False)
else:
    moved_paths = {}
    print("Исправление меток выключено. Чтобы включить, задай APPLY_LABEL_FIXES = True.")


In [ ]:
train_val_files = sorted(list(TRAIN_DIR.rglob('*.jpg')))
test_files = sorted(list(TEST_DIR.rglob('*.jpg')))
print("Всего изображений найдено:", len(train_val_files))
print("Изображений для теста:", len(test_files))


Кодировать имена персонажей в числовые метки класса и обратно будем при помощи `LabelEncoder`.

Для train выборки сформируем список текстовых меток всех изображений - имя родительской директории, которая одновременно является и именем персонажа. Зададим числовые метки классов нашего энкодера при помощи метода `fit`.

Далее будем применять метод `transform` для преобразования текстовых меток в числовые, и метод `inverse_transform` для преобразования числовых меток в текстовые.


In [ ]:
label_encoder = LabelEncoder()

train_val_labels = [path.parent.name for path in train_val_files]

label_encoder.fit(train_val_labels)


Разделим train выборку на обучающую и валидационнную части. Для того, чтобы персонажи были пропорционально представлены в обучающей и валидационнной подвыборках, применим стратификацию по меткам класса.


In [ ]:
from sklearn.model_selection import train_test_split

train_files, val_files = train_test_split(
    train_val_files,
    test_size=0.25,
    stratify=train_val_labels,
    random_state=42,
)


####Создадим Datasets и Dataloaders


In [ ]:
class SimpsonsDataset(Dataset):
    def __init__(self, files, label_encoder, mode):
        super().__init__()

        self.files = sorted(files)
        self.mode = mode

        if self.mode not in DATA_MODES:
            print(f"{self.mode} is not correct; correct modes: {DATA_MODES}")
            raise NameError

        self.label_encoder = label_encoder
        self.len_ = len(self.files)

        self.train_transform = v2.Compose([
            v2.Resize([256, 256]),
            v2.RandomResizedCrop(RESCALE_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomRotation(degrees=10),
            v2.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.03,
            ),
            v2.PILToTensor(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
        ])

        self.val_transform = v2.Compose([
            v2.Resize(RESCALE_SIZE),
            v2.PILToTensor(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
        ])

    def __len__(self):
        return self.len_

    def __getitem__(self, index):
        x = self.load_image(self.files[index])
        x = self.transform_images_to_tensors(x)

        if self.mode == "test":
            return x

        path = self.files[index]
        y = self.label_encoder.transform([path.parent.name]).item()
        return x, y

    def load_image(self, file):
        image = Image.open(file).convert("RGB")
        image.load()
        return image

    def transform_images_to_tensors(self, image):
        if self.mode == "train":
            return self.train_transform(image)
        return self.val_transform(image)


In [ ]:
train_dataset = SimpsonsDataset(train_files, label_encoder = label_encoder, mode='train')
val_dataset = SimpsonsDataset(val_files, label_encoder, mode='val')


In [ ]:
batch_size = 64


Посмотрим распределенние классов


In [ ]:
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def get_simpsons_counts(dataset):
    class_names = list(dataset.label_encoder.classes_)

    class_names_from_paths = [
        path.parent.name
        for path in dataset.files
    ]

    counts_by_class_name = Counter(class_names_from_paths)

    counts = [
        counts_by_class_name[class_name]
        for class_name in class_names
    ]

    return class_names, counts


def plot_train_val_distribution(train_dataset, val_dataset, title="Train vs Validation class distribution"):
    train_class_names, train_counts = get_simpsons_counts(train_dataset)
    val_class_names, val_counts = get_simpsons_counts(val_dataset)

    assert train_class_names == val_class_names, "Классы в train и val не совпадают"

    df = pd.DataFrame({
        "class_name": train_class_names,
        "train_count": train_counts,
        "val_count": val_counts,
    })

    df["total_count"] = df["train_count"] + df["val_count"]
    df = df.sort_values("total_count", ascending=False).reset_index(drop=True)

    x = np.arange(len(df))
    width = 0.42

    plt.figure(figsize=(18, 6))
    plt.bar(x - width / 2, df["train_count"], width, label="train")
    plt.bar(x + width / 2, df["val_count"], width, label="val")

    plt.xticks(x, df["class_name"], rotation=90)
    plt.xlabel("Character")
    plt.ylabel("Number of images")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # display(df[["class_name", "train_count", "val_count", "total_count"]])

    print("Train всего:", int(df["train_count"].sum()))
    print("Val всего:", int(df["val_count"].sum()))
    print("Всего:", int(df["total_count"].sum()))

    print("\nTrain min/max:", int(df["train_count"].min()), "/", int(df["train_count"].max()))
    print("Val min/max:", int(df["val_count"].min()), "/", int(df["val_count"].max()))

    return df


distribution_df = plot_train_val_distribution(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    title="Train vs Validation distribution by Simpsons character"
)


Большой дисбаланс классов, будем использовать WeightedRandomSampler


In [ ]:
train_numeric_labels = label_encoder.transform([
    path.parent.name for path in train_dataset.files
])

class_counts = np.bincount(
    train_numeric_labels,
    minlength=len(label_encoder.classes_)
)

class_weights_for_sampler = 1.0 / np.sqrt(np.maximum(class_counts, 1))

sample_weights = torch.DoubleTensor([
    class_weights_for_sampler[label]
    for label in train_numeric_labels
])

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    sampler=train_sampler,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

loaders = {
    "train": train_loader,
    "val": val_loader,
}

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Number of classes:", len(label_encoder.classes_))


Давайте посмотрим на наших героев внутри датасета.


Функции визуализации вынесены в `visualization.py`.


In [ ]:
from visualization import (
    imshow,
    show_images_from_loader,
    show_images_with_predictions,
)


In [ ]:
show_images_from_loader(n_rows=3, n_cols=4, loader=train_loader, label_encoder=label_encoder)


## Шаг 3. Построение нейросети



#### Модель


In [ ]:
# Вместо SimpleCnn используем EfficientNet
# Функции обучения, evaluate, predict, plot_history и сохранение чекпоинтов берём из utils.py.


In [ ]:
from torchvision import models
from torchvision.models import EfficientNet_V2_S_Weights
import torch.nn as nn

def create_model(n_classes):
    weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
    model = models.efficientnet_v2_s(weights=weights)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, n_classes)
    )

    return model

model_simple_cnn = create_model(len(label_encoder.classes_)).to(DEVICE)


## Шаг 4. Функции для работы с моделью


In [ ]:
import sys

utils_path = str(DRIVE_PROJECT_DIR)
if utils_path not in sys.path:
    sys.path.insert(0, utils_path)

from utils import (
    evaluate,
    load_trained_model_from_checkpoint,
    log_metrics_to_tensorboard,
    make_training_history,
    plot_history,
    predict,
    save_best_model,
    save_history_json,
    save_model_and_history,
    save_training_state,
    train_CNN,
)


In [ ]:
history = make_training_history()


## Шаг 5. Применение модели к данным


#### Применение модели к данным


In [ ]:
# Также будем использовать CrossEntropyLoss с весами

class_weights = 1.0 / np.sqrt(np.maximum(class_counts, 1))
class_weights = class_weights / class_weights.mean()

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

# Scheduler нельзя создавать до optimizer. Используем фабрику и вызываем ее
# после создания optimizer для конкретного этапа обучения.
def make_cosine_scheduler(optimizer, t_max, eta_min=1e-6):
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=t_max,
        eta_min=eta_min,
    )


In [ ]:
# Этап 1: обучаем только новый классификатор EfficientNet.

for param in model_simple_cnn.parameters():
    param.requires_grad = False

for param in model_simple_cnn.classifier.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss(label_smoothing=0.02)

optimizer = torch.optim.AdamW(
    model_simple_cnn.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

history = train_CNN(
    model=model_simple_cnn,
    num_epochs=3,
    dataloaders=loaders,
    optimizer=optimizer,
    loss_func=criterion,
    device=DEVICE,
    val_every_steps=100,
    save_dir=EFFICIENTNET_STAGE1_DIR,
    writer=writer,
)

# Этап 2: размораживаем всю EfficientNet и дообучаем его маленьким learning rate.

for param in model_simple_cnn.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {"params": model_simple_cnn.features.parameters(), "lr": 1e-5},
        {"params": model_simple_cnn.classifier.parameters(), "lr": 3e-4},
    ],
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.02
)

stage2_scheduler = make_cosine_scheduler(optimizer, t_max=10)

history = train_CNN(
    model=model_simple_cnn,
    num_epochs=10,
    dataloaders=loaders,
    optimizer=optimizer,
    loss_func=criterion,
    device=DEVICE,
    val_every_steps=100,
    save_dir=EFFICIENTNET_STAGE2_DIR,
    writer=writer,
    history=history,
    scheduler=stage2_scheduler,
)

plot_history(history)


In [ ]:
from sklearn.metrics import classification_report

model = model_simple_cnn
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        logits = model(images)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

print(classification_report(
    all_targets,
    all_preds,
    target_names=label_encoder.classes_,
    digits=4
))


Посмотрим где нейросеть ошибается, но при этом очень уверена, что права - чтобы исправить неправильные метки


In [ ]:
from label_audit import (
    build_train_audit_dataset,
    find_wrong_label_candidates,
    run_label_audit,
    select_suspicious_examples,
    show_class_pair_confusions,
    show_review_examples,
    show_suspicious_examples,
)


In [ ]:
review_csv_path = LABEL_AUDIT_DIR / "suspicious_manual_review.csv"
print("Label audit directory:", LABEL_AUDIT_DIR)


In [ ]:
audit_dataset, df_all, df_suspicious = run_label_audit(
    model=model_simple_cnn,
    train_files=train_val_files,
    label_encoder=label_encoder,
    dataset_cls=SimpsonsDataset,
    device=DEVICE,
    batch_size=64,
    num_workers=0,
    min_confidence=0.90,
    min_margin=0.20,
    top_n=300,
    save_all_csv_path=LABEL_AUDIT_DIR / "all_predictions.csv",
    save_review_csv_path=LABEL_AUDIT_DIR / "suspicious_labels_review.csv",
)


Функции аудита меток вынесены в `label_audit.py`.


In [ ]:
review_df = df_suspicious.copy()
review_df["action"] = ""          # варианты: "move", "keep", "skip"
review_df["correct_label"] = ""   # например: "homer_simpson"

review_csv_path = LABEL_AUDIT_DIR / "suspicious_manual_new.csv"
review_df.to_csv(review_csv_path, index=False)

show_review_examples(
    review_df=review_df,
    dataset=audit_dataset,
    start_pos=0,
    n_rows=4,
    n_cols=3,
)


Сделаем визуализацию,  чтобы посмотреть насколько сеть уверена в своих ответах.


Функция показа предсказаний вынесена в `visualization.py` как `show_images_with_predictions`.


In [ ]:
show_images_with_predictions(
    n_rows=3,
    n_cols=3,
    dataset=val_dataset,
    model=model_simple_cnn,
    label_encoder=label_encoder,
    device=DEVICE,
)

## Шаг 6. Submit на Kaggle


Создадим loader для тестовых данных


In [ ]:
test_dataset = SimpsonsDataset(test_files, label_encoder = label_encoder, mode="test")
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=64)


Воспользуемся функцией predict, которая возвращает предсказанные числовые метки для всех объектов в лоадере.


Получем предсказание меток классов для тестовых данных:


In [ ]:
if BEST_MODEL_PATH.exists():
    model_simple_cnn.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    model_simple_cnn.to(DEVICE)
    model_simple_cnn.eval()
else:
    print(f"Best model checkpoint not found: {BEST_MODEL_PATH}. Using current model state.")

predicted_numeric_labels = predict(model_simple_cnn, test_loader, device=DEVICE)
predicted_numeric_labels = predicted_numeric_labels.numpy()


и преобразуем их в текстовые метки:


In [ ]:
predicted_text_labels = label_encoder.inverse_transform(predicted_numeric_labels)


Загрузим пример файла для загрузки на Kaggle (проверьте путь, по которому у вас лежит файл sample_submission.csv и при необходимости скорректируйте путь в коде ниже):


In [ ]:
import pandas as pd
sample_submission = pd.read_csv("/content/sample_submission.csv")
sample_submission.head(10)


In [ ]:
my_submission = pd.DataFrame({'Id': [path.name for path in test_files], 'Expected': predicted_text_labels})
my_submission.head(10)


In [ ]:
# TODO : сделайте сабмит (это важно, если Вы не справляетесь, но дошли до этой ячейки, то сообщите в чат и Вам помогут)


In [ ]:
my_submission.to_csv('simple_cnn_baseline.csv', index=False)
